# Phase 10: Deep Temporal Dynamics & Advanced Calibration

In this ultimate notebook, we implement the architectural vision of **Track A** (Deep Temporal Features) and **Track B** (Logit Stacking & Calibration Audit).

### Track A: Deep Temporal Features
We transition from basic slopes to:
1. **Drawdowns:** Measuring peak-to-trough collapse (how much money did the user lose relative to their historical maximum).
2. **Multi-Horizon Slopes:** Capturing `3-month` vs `6-month` trends and measuring *acceleration*.
3. **Customer-Normalized Z-Scores:** How abnormal is the user's current behavior relative to their own standard deviation?
4. **Deterioration Streaks:** Counting consecutive months of decline.

### Track B: The Calibration Audit
We abandon Isotonic regression in favor of a robust audit evaluating:
1. **Logit-Space Stacking:** Feeding unconstrained evidence to the Meta-Learner.
2. **Platt Scaling (Sigmoid)**
3. **Temperature Scaling**
4. **Probability Shrinkage**

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import mlflow
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('../')
from src.metrics import zindi_score

mlflow.set_tracking_uri('sqlite:///../mlflow.db')
mlflow.set_experiment('Zindi_Liquidity_Stress')

/Users/USER/Desktop/zindi/ml_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='/Users/USER/Desktop/zindi/notebooks/mlruns/1', creation_time=1788129250583, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788129250583, lifecycle_stage='active', name='Zindi_Liquidity_Stress', tags={}, trace_location=None, workspace='default'>

## 1. Track A: Engineering Deep Temporal Features

In [ ]:
train_df = pd.read_parquet('../data/features/train_features.parquet', engine='fastparquet')
test_df = pd.read_parquet('../data/features/test_features.parquet', engine='fastparquet')
raw_train = pd.read_csv('../data/Train.csv')
raw_test = pd.read_csv('../data/Test.csv')

target = raw_train['liquidity_stress_next_30d']

def engineer_deep_temporal(df):
    df = df.copy()
    
    bal_cols = ['m6_daily_avg_bal', 'm5_daily_avg_bal', 'm4_daily_avg_bal', 'm3_daily_avg_bal', 'm2_daily_avg_bal', 'm1_daily_avg_bal']
    bal_matrix = df[bal_cols].values
    
    # 1. Drawdown Features (Peak to Trough)
    peak_6m = np.max(bal_matrix, axis=1)
    df['bal_drawdown'] = (df['m1_daily_avg_bal'] - peak_6m) / (peak_6m + 1)
    
    # 2. Multi-Horizon Slopes & Acceleration
    x_6m = np.array([6, 5, 4, 3, 2, 1])
    x_3m = np.array([3, 2, 1])
    
    def calc_slope_vectorized(matrix, x_arr):
        x_mean = np.mean(x_arr)
        x_diff = x_arr - x_mean
        denominator = np.sum(x_diff**2)
        return np.sum(x_diff * (matrix - np.mean(matrix, axis=1)[:, None]), axis=1) / denominator

    df['bal_slope_6m'] = calc_slope_vectorized(bal_matrix, x_6m)
    df['bal_slope_3m'] = calc_slope_vectorized(bal_matrix[:, -3:], x_3m)
    df['bal_slope_acceleration'] = df['bal_slope_3m'] - df['bal_slope_6m']
    
    # 3. Customer-Normalized Z-Scores (Distance from Normal)
    mean_6m = np.mean(bal_matrix, axis=1)
    std_6m = np.std(bal_matrix, axis=1)
    df['bal_zscore'] = (df['m1_daily_avg_bal'] - mean_6m) / (std_6m + 1e-5)
    
    # 4. Deterioration Streaks (Consecutive Declines)
    # A boolean matrix where True means the balance dropped from the previous month
    drops = np.diff(bal_matrix, axis=1) < 0
    
    # We want to count consecutive True values starting from the most recent (m1) going backwards
    streak_counts = np.zeros(len(df))
    for i in range(4, -1, -1): # Start at most recent transition (m2 -> m1)
        streak_counts = np.where(drops[:, i], streak_counts + 1, 0)
    df['bal_decline_streak'] = streak_counts

    # Add SHAP Zero-Balance Flags
    df['is_m1_zero'] = (df['m1_daily_avg_bal'] < 1000).astype(int)
    df['is_m2_zero'] = (df['m2_daily_avg_bal'] < 1000).astype(int)
    
    # Drop Noise
    noisy_features = ['outflow_volatility', 'runway_months']
    df = df.drop(columns=[col for col in noisy_features if col in df.columns])
    
    return df

print("Engineering Deep Temporal Features...")
X = engineer_deep_temporal(train_df)
X = X.drop(columns=['ID', 'liquidity_stress_next_30d'])

X_test = engineer_deep_temporal(test_df)
X_test = X_test.drop(columns=['ID'])

# One-Hot Encoding
cat_cols = X.select_dtypes(include=['object', 'category']).columns
if len(cat_cols) > 0:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
    X_test = X_test.reindex(columns=X.columns, fill_value=0)

X = X.astype({col: int for col in X.select_dtypes(include=bool).columns})
X_test = X_test.astype({col: int for col in X_test.select_dtypes(include=bool).columns})
print(f"Final Dataset Shape: {X.shape}")

Engineering Deep Temporal Features...
Final Dataset Shape: (40000, 234)


## 2. Generate Base OOF Predictions

In [3]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))
oof_cb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

test_lgb = np.zeros(len(X_test))
test_cb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))

lgb_params = {'learning_rate': 0.07000857701607659, 'num_leaves': 27, 'max_depth': 8, 'min_child_samples': 71, 'subsample': 0.7227032899255299, 'colsample_bytree': 0.8012769608390673, 'n_estimators': 500, 'random_state': 42, 'verbosity': -1}
cb_params = {'learning_rate': 0.19775545782423468, 'depth': 5, 'l2_leaf_reg': 0.5290629979609613, 'subsample': 0.6633141772092477, 'iterations': 500, 'random_seed': 42, 'verbose': False}
xgb_params = {'learning_rate': 0.05612762846223619, 'max_depth': 9, 'min_child_weight': 14, 'subsample': 0.7984977991980712, 'colsample_bytree': 0.873957633262947, 'n_estimators': 500, 'random_state': 42, 'verbosity': 0}

print("Training Base Models...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, target)):
    X_train, y_train = X.iloc[train_idx], target.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], target.iloc[val_idx]
    
    model_lgb = lgb.LGBMClassifier(**lgb_params)
    model_lgb.fit(X_train, y_train)
    oof_lgb[val_idx] = model_lgb.predict_proba(X_val)[:, 1]
    test_lgb += model_lgb.predict_proba(X_test)[:, 1] / N_FOLDS
    
    model_cb = cb.CatBoostClassifier(**cb_params)
    model_cb.fit(X_train, y_train)
    oof_cb[val_idx] = model_cb.predict_proba(X_val)[:, 1]
    test_cb += model_cb.predict_proba(X_test)[:, 1] / N_FOLDS

    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X_train, y_train)
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    test_xgb += model_xgb.predict_proba(X_test)[:, 1] / N_FOLDS
    
    print(f"Fold {fold+1} Done.")

Training Base Models...
Fold 1 Done.
Fold 2 Done.
Fold 3 Done.
Fold 4 Done.
Fold 5 Done.


## 3. Track B: Logit Stacking & The Calibration Audit
Instead of stacking probabilities (which are bounded 0 to 1), we convert them to logits (unbounded log-odds) before feeding them to Logistic Regression. This provides an elegant, naturally calibrated mathematical combination.

Then, we evaluate how different Probability Regularization techniques affect our LogLoss.

In [4]:
# Convert Probabilities to Logits
def to_logits(p):
    p_clip = np.clip(p, 1e-15, 1 - 1e-15)
    return np.log(p_clip / (1 - p_clip))

X_meta_logit = np.column_stack([to_logits(oof_lgb), to_logits(oof_cb), to_logits(oof_xgb)])
X_test_meta_logit = np.column_stack([to_logits(test_lgb), to_logits(test_cb), to_logits(test_xgb)])

# Logistic Regression on Logits (The optimal Platt-Scaled Stacker)
stacker = LogisticRegression(penalty='l2', C=1.0)
stacker.fit(X_meta_logit, target)

print(f"\nLearned Logit Weights: LGBM: {stacker.coef_[0][0]:.3f}, CB: {stacker.coef_[0][1]:.3f}, XGB: {stacker.coef_[0][2]:.3f}")

stacked_oof = stacker.predict_proba(X_meta_logit)[:, 1]
stacked_test = stacker.predict_proba(X_test_meta_logit)[:, 1]

# --------------------------------------------------------
# THE CALIBRATION AUDIT
# --------------------------------------------------------
print("\n=== CALIBRATION AUDIT LEADERBOARD ===")

# 1. Baseline Logit Stack
base_loss, base_auc = zindi_score(target, stacked_oof)
print(f"[1] Baseline Logit Stacking | LogLoss: {base_loss:.4f} | AUC: {base_auc:.4f}")

# 2. Probability Shrinkage (Pulling toward base rate)
base_rate = np.mean(target)
for shrink in [0.01, 0.05, 0.10]:
    shrunk_oof = (1 - shrink) * stacked_oof + shrink * base_rate
    loss, auc = zindi_score(target, shrunk_oof)
    print(f"[2] Shrinkage (lambda={shrink:.2f})  | LogLoss: {loss:.4f} | AUC: {auc:.4f}")

# 3. Temperature Scaling (Softening overconfident logits)
stacked_logits_out = to_logits(stacked_oof)
for T in [1.1, 1.25, 1.5]:
    temp_oof = 1 / (1 + np.exp(-stacked_logits_out / T))
    loss, auc = zindi_score(target, temp_oof)
    print(f"[3] Temperature (T={T:.2f})    | LogLoss: {loss:.4f} | AUC: {auc:.4f}")

with mlflow.start_run(run_name='Deep_Temporal_Logit_Stack'):
    mlflow.log_metric('cv_logloss', base_loss)
    mlflow.log_metric('cv_auc', base_auc)


Learned Logit Weights: LGBM: 0.468, CB: 0.325, XGB: 0.176

=== CALIBRATION AUDIT LEADERBOARD ===
--- Model Evaluation ---
Log Loss: 0.2606
ROC-AUC: 0.8957
------------------------
[1] Baseline Logit Stacking | LogLoss: 0.2606 | AUC: 0.8957
--- Model Evaluation ---
Log Loss: 0.2608
ROC-AUC: 0.8957
------------------------
[2] Shrinkage (lambda=0.01)  | LogLoss: 0.2608 | AUC: 0.8957
--- Model Evaluation ---
Log Loss: 0.2623
ROC-AUC: 0.8957
------------------------
[2] Shrinkage (lambda=0.05)  | LogLoss: 0.2623 | AUC: 0.8957
--- Model Evaluation ---
Log Loss: 0.2651
ROC-AUC: 0.8957
------------------------
[2] Shrinkage (lambda=0.10)  | LogLoss: 0.2651 | AUC: 0.8957
--- Model Evaluation ---
Log Loss: 0.2619
ROC-AUC: 0.8957
------------------------
[3] Temperature (T=1.10)    | LogLoss: 0.2619 | AUC: 0.8957
--- Model Evaluation ---
Log Loss: 0.2673
ROC-AUC: 0.8957
------------------------
[3] Temperature (T=1.25)    | LogLoss: 0.2673 | AUC: 0.8957
--- Model Evaluation ---
Log Loss: 0.2821

## Overfitting Check

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss

def check_overfitting():
    try:
        models_to_check = {}
        if 'model_lgb' in globals(): models_to_check['LightGBM'] = model_lgb
        if 'model_cb' in globals(): models_to_check['CatBoost'] = model_cb
        if 'model_xgb' in globals(): models_to_check['XGBoost'] = model_xgb
        if 'model' in globals() and 'LGBM' in str(type(model)): models_to_check['LightGBM'] = model
        
        if not models_to_check:
            print("No standard models found in memory to check.")
            return
            
        if 'X_train' not in globals() or 'X_val' not in globals():
            print("X_train or X_val not found in memory.")
            return
            
        train_losses = []
        val_losses = []
        names = []
        
        for name, m in models_to_check.items():
            train_p = m.predict_proba(X_train)[:, 1]
            val_p = m.predict_proba(X_val)[:, 1]
            train_losses.append(log_loss(y_train, train_p))
            val_losses.append(log_loss(y_val, val_p))
            names.append(name)
            
        x = range(len(names))
        width = 0.35
        
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.bar([i - width/2 for i in x], train_losses, width, label='Train Loss', color='#3498db')
        ax.bar([i + width/2 for i in x], val_losses, width, label='Val Loss', color='#e74c3c')
        
        ax.set_ylabel('Log Loss')
        ax.set_title('Overfitting Check: Train vs Validation Loss (Last Fold)')
        ax.set_xticks(x)
        ax.set_xticklabels(names)
        ax.legend()
        plt.show()
        
        for i, name in enumerate(names):
            print(f"{name} - Train: {train_losses[i]:.4f}, Val: {val_losses[i]:.4f}, Gap: {val_losses[i] - train_losses[i]:.4f}")
            
    except Exception as e:
        print(f"Could not run overfitting check: {e}")

check_overfitting()